# Entrenamiento Stuff+ desde Google Drive

Este cuaderno lee desde Drive `stuff_model_df.parquet`, `entrenar_stuff_plus.py` y `requirements-hackathon.txt`; copia el código y el Parquet al disco temporal de Colab, entrena los siete modelos y exporta los resultados de vuelta a Drive.

Sube la carpeta del proyecto a `Mi unidad`. En la primera celda de código, ajusta `PROJECT_DIR` si la carpeta tiene otro nombre. El algoritmo solo consume el Parquet; no necesita los archivos `.pkl` ni `.rds`. Ejecuta las celdas en orden y mantén la sesión activa hasta terminar.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

# Cambia esta ruta si subiste la carpeta con otro nombre o dentro de otra carpeta.
PROJECT_DIR = Path('/content/drive/MyDrive/Diablos - Hackathon 2026')
PARQUET_IN_DRIVE = PROJECT_DIR / 'stuff_model_df.parquet'
SCRIPT_IN_DRIVE = PROJECT_DIR / 'entrenar_stuff_plus.py'
REQUIREMENTS_IN_DRIVE = PROJECT_DIR / 'requirements-hackathon.txt'

for required_file in (PARQUET_IN_DRIVE, SCRIPT_IN_DRIVE, REQUIREMENTS_IN_DRIVE):
    if not required_file.is_file():
        raise FileNotFoundError(f'No encontré {required_file}. Corrige PROJECT_DIR y vuelve a ejecutar esta celda.')

print('Carpeta de Drive:', PROJECT_DIR)
print(f'Parquet: {PARQUET_IN_DRIVE.stat().st_size / (1024**2):.1f} MB')

In [ ]:
import subprocess, sys

# Instala las dependencias declaradas por el proyecto.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS_IN_DRIVE)], check=True)
print('Dependencias instaladas.')

In [ ]:
# Copia los archivos grandes al disco local de la sesión para entrenar desde /content.
WORK_DIR = Path('/content/stuff_plus_run')
WORK_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_LOCAL = WORK_DIR / 'stuff_model_df.parquet'
SCRIPT_LOCAL = WORK_DIR / 'entrenar_stuff_plus.py'
shutil.copy2(PARQUET_IN_DRIVE, PARQUET_LOCAL)
shutil.copy2(SCRIPT_IN_DRIVE, SCRIPT_LOCAL)
print('Archivos preparados en', WORK_DIR)

In [ ]:
# Entrena con las opciones predeterminadas. Los resultados se generan primero en el disco de Colab.
OUTPUT_LOCAL = WORK_DIR / 'stuff_plus_output'
cmd = [
    sys.executable, '-u', str(SCRIPT_LOCAL),
    '--parquet', str(PARQUET_LOCAL),
    '--salida', str(OUTPUT_LOCAL),
]
subprocess.run(cmd, cwd=WORK_DIR, check=True)

In [ ]:
import json
from datetime import datetime
from IPython.display import display
import pandas as pd

# Exporta cada ejecución a una carpeta nueva de Drive para conservar resultados anteriores.
run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DRIVE = PROJECT_DIR / f'stuff_plus_output_{run_id}'
shutil.copytree(OUTPUT_LOCAL, OUTPUT_DRIVE)

metrics = json.loads((OUTPUT_DRIVE / 'metricas_validacion.json').read_text(encoding='utf-8'))
display(pd.DataFrame(metrics))
print('Resultados exportados a Drive:', OUTPUT_DRIVE)
print('Incluye metricas_validacion.json, scores_pitch.parquet y la carpeta modelos.')